# Lab 21: Decision Trees and Forests (graded)

You are starting a company that grows and sells wild mushrooms. 
- Since not all mushrooms are edible, you'd like to be able to tell whether a given mushroom is edible or poisonous based on it's physical attributes
- You have some existing data that you can use for this task. It is already one-hot coded so all features are binary.

Goal: identify which mushrooms can be sold safely

NB: The dataset used is for illustrative purposes only. It is not meant to be a guide on identifying edible mushrooms.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from public_tests import *
from lib.utils import *

%matplotlib inline

## Overview of the process

* Start with all examples at the root node
* Calculate information gain for splitting on all possible features, and pick the one with the highest information gain
* Split dataset according to the selected feature, and create left and right branches of the tree
* Keep repeating splitting process until stopping criteria is met

In this lab, you'll implement the following functions, which will let you split a node into left and right branches using the feature with the highest information gain
* Calculate the entropy at a node
* Split the dataset at a node into left and right branches based on a given feature
* Calculate the information gain from splitting on a given feature
* Choose the feature that maximizes information gain

We'll then use the helper functions you've implemented to build a decision tree by repeating the splitting process until the stopping criteria is met. 

For this lab, the stopping criteria we've chosen is setting a maximum depth of 2.

## Load data

X_train contains three features for each example
* Brown Color (A value of 1 indicates "Brown" cap color and 0 indicates "Red" cap color)
* Tapering Shape (A value of 1 indicates "Tapering Stalk Shape" and 0 indicates "Enlarging" stalk shape)
* Solitary (A value of 1 indicates "Yes" and 0 indicates "No")

y_train is whether the mushroom is edible
* y = 1 indicates edible
* y = 0 indicates poisonous

In [4]:
X_train = np.array([[1,1,1],[1,0,1],[1,0,0],[1,0,0],[1,1,1],[0,1,1],[0,0,0],[1,0,1],[0,1,0],[1,0,0]])
y_train = np.array([1,1,0,0,1,0,0,1,1,0])

In [5]:
# inspect data

print("First element of X_train:\n", X_train[0])
print("First few elements of X_train:\n", X_train[:5])
print("Type of X_train:",type(X_train))

print("\nFirst few elements of y_train:", y_train[:5])
print("Type of y_train:",type(y_train))

print ('\nThe shape of X_train is:', X_train.shape)
print ('The shape of y_train is: ', y_train.shape)
print ('Number of training examples (m):', len(X_train))

First element of X_train:
 [1 1 1]
First few elements of X_train:
 [[1 1 1]
 [1 0 1]
 [1 0 0]
 [1 0 0]
 [1 1 1]]
Type of X_train: <class 'numpy.ndarray'>

First few elements of y_train: [1 1 0 0 1]
Type of y_train: <class 'numpy.ndarray'>

The shape of X_train is: (10, 3)
The shape of y_train is:  (10,)
Number of training examples (m): 10


## Calculate entropy

In [19]:
# UNQ_C1
# GRADED FUNCTION: compute_entropy

def compute_entropy(y):
    """
    Computes the entropy for 
    
    Args:
       y (ndarray): Numpy array indicating whether each example at a node is
           edible (`1`) or poisonous (`0`)
       
    Returns:
        entropy (float): Entropy at that node
        
    """
    # You need to return the following variables correctly
    entropy = 0.
    
    ### START CODE HERE ###
    
    # Must check if the data at a node is empty
    if len(y) == 0:
        return entropy
  
    p = sum(y)/len(y)

    # Entropy = 0 when the selection is completely pure...  p = 0 or p = 1
    # Entropy = 1 when the split of elements is 50/50. That is the least pure situation possible.
    if p != 0 and p != 1:
        entropy =  -p * np.log2(p) - (1 - p)*np.log2(1 - p)          
            
    ### END CODE HERE ###        
    
    return entropy

In [20]:
def compute_entropy_test(target):
    y = np.array([1] * 10)
    result = target(y)
    
    assert result == 0, "Entropy must be 0 with array of ones"
    
    y = np.array([0] * 10)
    result = target(y)
    
    assert result == 0, "Entropy must be 0 with array of zeros"
    
    y = np.array([0] * 12 + [1] * 12)
    result = target(y)
    
    assert result == 1, "Entropy must be 1 with same ammount of ones and zeros"
    
    y = np.array([1, 0, 1, 0, 1, 1, 1, 0, 1])
    assert np.isclose(target(y), 0.918295, atol=1e-6), "Wrong value. Something between 0 and 1"
    assert np.isclose(target(-y + 1), target(y), atol=1e-6), "Wrong value"
    
    print("\033[92m All tests passed. ")


In [21]:
# Compute entropy at the root node (i.e. with all examples)
# Since we have 5 edible and 5 non-edible mushrooms, the entropy should be 1"

print("Entropy at root node: ", compute_entropy(y_train)) 

# UNIT TESTS
def compute_entropy_test(target):
    y = np.array([1] * 10)
    result = target(y)
    
    assert result == 0, "Entropy must be 0 with array of ones"
    
    y = np.array([0] * 10)
    result = target(y)
    
    assert result == 0, "Entropy must be 0 with array of zeros"
    
    y = np.array([0] * 12 + [1] * 12)
    result = target(y)
    
    assert result == 1, "Entropy must be 1 with same ammount of ones and zeros"
    
    y = np.array([1, 0, 1, 0, 1, 1, 1, 0, 1])
    assert np.isclose(target(y), 0.918295, atol=1e-6), "Wrong value. Something between 0 and 1"
    assert np.isclose(target(-y + 1), target(y), atol=1e-6), "Wrong value"
    
    print("\033[92m All tests passed. ")

compute_entropy_test(compute_entropy)

Entropy at root node:  1.0
 All tests passed. 


## Split the datasets

In [22]:
# UNQ_C2
# GRADED FUNCTION: split_dataset

def split_dataset(X, node_indices, feature):
    """
    Splits the data at the given node into
    left and right branches
    
    Args:
        X (ndarray):             Data matrix of shape(n_samples, n_features)
        node_indices (list):     List containing the active indices. I.e, the samples being considered at this step.
        feature (int):           Index of feature to split on
    
    Returns:
        left_indices (list):     Indices with feature value == 1
        right_indices (list):    Indices with feature value == 0
    """
    
    # You need to return the following variables correctly
    left_indices = []
    right_indices = []
    
    ### START CODE HERE ###
    for i in node_indices:

        # if the feature is 'hot'/true/1... then it goes in left group
        if X[i][feature] == 1:
            left_indices.append(i)

        # else it goes in right group
        else:
            right_indices.append(i)
            
    ### END CODE HERE ###
        
    return left_indices, right_indices

In [23]:
# Feel free to play around with these variables
# The dataset only has three features, so this value can be 0 (Brown Cap), 1 (Tapering Stalk Shape) or 2 (Solitary)


print(X_train)

# Case 1

root_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
print(root_indices)

feature = 0
left_indices0, right_indices0 = split_dataset(X_train, root_indices, feature)
print("Case 1; feature 0:")
print("Left indices: ", left_indices0)
print("Right indices: ", right_indices0)

feature = 1
left_indices1, right_indices1 = split_dataset(X_train, root_indices, feature)
print("Case 1; feature 1:")
print("Left indices: ", left_indices1)
print("Right indices: ", right_indices1)

feature = 2
left_indices2, right_indices2 = split_dataset(X_train, root_indices, feature)
print("Case 1; feature 2:")
print("Left indices: ", left_indices2)
print("Right indices: ", right_indices2)

# Visualize the split 
# generate_split_viz(root_indices, left_indices, right_indices, feature)

print()

# Case 2

root_indices_subset = [0, 2, 4, 6, 8]
print(root_indices_subset)

feature = 0
left_indices_partial, right_indices_partial = split_dataset(X_train, root_indices_subset, feature)

print("CASE 2:")
print("Left indices: ", left_indices_partial)
print("Right indices: ", right_indices_partial)

# Visualize the split 
# generate_split_viz(root_indices_subset, left_indices, right_indices, feature)

# UNIT TESTS    
split_dataset_test(split_dataset)

[[1 1 1]
 [1 0 1]
 [1 0 0]
 [1 0 0]
 [1 1 1]
 [0 1 1]
 [0 0 0]
 [1 0 1]
 [0 1 0]
 [1 0 0]]
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Case 1; feature 0:
Left indices:  [0, 1, 2, 3, 4, 7, 9]
Right indices:  [5, 6, 8]
Case 1; feature 1:
Left indices:  [0, 4, 5, 8]
Right indices:  [1, 2, 3, 6, 7, 9]
Case 1; feature 2:
Left indices:  [0, 1, 4, 5, 7]
Right indices:  [2, 3, 6, 8, 9]

[0, 2, 4, 6, 8]
CASE 2:
Left indices:  [0, 2, 4]
Right indices:  [6, 8]
 All tests passed.


## Calculate information gain

In [26]:
# UNQ_C3
# GRADED FUNCTION: compute_information_gain

def compute_information_gain(X, y, node_indices, feature):
    
    """
    Compute the information of splitting the node on a given feature
    
    Args:
        X (ndarray):            Data matrix of shape(n_samples, n_features)
        y (array like):         list or ndarray with n_samples containing the target variable
        node_indices (ndarray): List containing the active indices. I.e, the samples being considered in this step.
        feature (int):           Index of feature to split on
   
    Returns:
        cost (float):        Cost computed
    
    """    
    # Split dataset
    left_indices, right_indices = split_dataset(X, node_indices, feature)
    
    # Some useful variables
    X_node, y_node = X[node_indices], y[node_indices]
    X_left, y_left = X[left_indices], y[left_indices]
    X_right, y_right = X[right_indices], y[right_indices]
    
    # You need to return the following variables correctly
    information_gain = 0
    
    ### START CODE HERE ###
    
    entropy_before_split = compute_entropy(y_node)

    # entropy after the split
    w_left = len(left_indices)/len(node_indices)
    w_right = len(right_indices)/len(node_indices)
    weighted_entropy_after_split = (w_left * compute_entropy(y_left)) + (w_right * compute_entropy(y_right))

    # information gain is the change in entropy
    information_gain = entropy_before_split - weighted_entropy_after_split
    
    ### END CODE HERE ###  
    
    return information_gain

In [28]:
info_gain0 = compute_information_gain(X_train, y_train, root_indices, feature=0)
print("Information Gain from splitting the root on brown cap: ", info_gain0)

info_gain1 = compute_information_gain(X_train, y_train, root_indices, feature=1)
print("Information Gain from splitting the root on tapering stalk shape: ", info_gain1)

info_gain2 = compute_information_gain(X_train, y_train, root_indices, feature=2)
print("Information Gain from splitting the root on solitary: ", info_gain2)

# Expected output
# Information Gain from splitting the root on brown cap:  0.034851554559677034
# Information Gain from splitting the root on tapering stalk shape:  0.12451124978365313
# Information Gain from splitting the root on solitary:  0.2780719051126377

# UNIT TESTS
compute_information_gain_test(compute_information_gain)

Information Gain from splitting the root on brown cap:  0.034851554559677034
Information Gain from splitting the root on tapering stalk shape:  0.12451124978365313
Information Gain from splitting the root on solitary:  0.2780719051126377
 All tests passed.


### Conclusion

Spliting on feature at index 2 (solitary?) provides the greatest gain in entropy so we should select that feature for the root node.

### Automate selection of best feature for splitting

In [31]:
# UNQ_C4
# GRADED FUNCTION: get_best_split

def get_best_split(X, y, node_indices):   
    """
    Returns the optimal feature and threshold value
    to split the node data 
    
    Args:
        X (ndarray):            Data matrix of shape(n_samples, n_features)
        y (array like):         list or ndarray with n_samples containing the target variable
        node_indices (ndarray): List containing the active indices. I.e, the samples being considered in this step.

    Returns:
        best_feature (int):     The index of the best feature to split
    """    
    
    # Some useful variables
    num_features = X.shape[1]
    
    # You need to return the following variables correctly
    best_feature = -1
    
    ### START CODE HERE ###

    # early return -1 if the selection is already pure
    current_entropy = compute_entropy(y[node_indices])
    if current_entropy == 0:
        return best_feature
        
    
    # below executes for selection that is not yet pure
    maximum = float('-inf')

    # for each feature
    for i in range(num_features):
        info_gain = compute_information_gain(X, y, node_indices, i)

        # This will execute at least one time because maximum is set at negative infinity
        if info_gain > maximum:
            maximum = info_gain
            best_feature = i   
            
    ### END CODE HERE ##    
   
    return best_feature

In [32]:
best_feature = get_best_split(X_train, y_train, root_indices)
print("Best feature to split on: %d" % best_feature)

# UNIT TESTS
get_best_split_test(get_best_split)

Best feature to split on: 2
 All tests passed.


## Building a tree (not graded)

In [33]:
# Not graded
tree = []

def build_tree_recursive(X, y, node_indices, branch_name, max_depth, current_depth):
    """
    Build a tree using the recursive algorithm that split the dataset into 2 subgroups at each node.
    This function just prints the tree.
    
    Args:
        X (ndarray):            Data matrix of shape(n_samples, n_features)
        y (array like):         list or ndarray with n_samples containing the target variable
        node_indices (ndarray): List containing the active indices. I.e, the samples being considered in this step.
        branch_name (string):   Name of the branch. ['Root', 'Left', 'Right']
        max_depth (int):        Max depth of the resulting tree. 
        current_depth (int):    Current depth. Parameter used during recursive call.
   
    """ 

    # Maximum depth reached - stop splitting
    if current_depth == max_depth:
        formatting = " "*current_depth + "-"*current_depth
        print(formatting, "%s leaf node with indices" % branch_name, node_indices)
        return
   
    # Otherwise, get best split and split the data
    # Get the best feature and threshold at this node
    best_feature = get_best_split(X, y, node_indices) 
    
    formatting = "-"*current_depth
    print("%s Depth %d, %s: Split on feature: %d" % (formatting, current_depth, branch_name, best_feature))
    
    # Split the dataset at the best feature
    left_indices, right_indices = split_dataset(X, node_indices, best_feature)
    tree.append((left_indices, right_indices, best_feature))
    
    # continue splitting the left and the right child. Increment current depth
    build_tree_recursive(X, y, left_indices, "Left", max_depth, current_depth+1)
    build_tree_recursive(X, y, right_indices, "Right", max_depth, current_depth+1)

In [35]:
build_tree_recursive(X_train, y_train, root_indices, "Root", max_depth=2, current_depth=0)
# generate_tree_viz(root_indices, y_train, tree)

 Depth 0, Root: Split on feature: 2
- Depth 1, Left: Split on feature: 0
  -- Left leaf node with indices [0, 1, 4, 7]
  -- Right leaf node with indices [5]
- Depth 1, Right: Split on feature: 1
  -- Left leaf node with indices [8]
  -- Right leaf node with indices [2, 3, 6, 9]
